# SHIPP Stage 5A — Silver Listings Development

**Method:** Spark Window Current-State Reconstruction

```text
Bronze listing history
    ↓
partition by listing_id
    ↓
order by _pg_lsn DESC, _sort_by DESC
    ↓
select rn = 1
    ↓
retain current-row change types
    ↓
quality checks
    ↓
silver_listings
```

This notebook is for development and validation before promoting logic to
`data_pipeline/silver/silver_listings.py`.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE = "bootcamp_students.shipp_bronze.lb_listings_history"
TARGET = "bootcamp_students.shipp_silver.silver_listings"

bronze = spark.table(SOURCE)
print("Bronze listing history rows:", bronze.count())
bronze.printSchema()


In [ ]:
# Inspect actual change types before transforming.
display(
    bronze.groupBy("_pg_change_type")
    .count()
    .orderBy("_pg_change_type")
)


In [ ]:
latest_window = Window.partitionBy("listing_id").orderBy(
    F.col("_pg_lsn").desc(),
    F.col("_sort_by").desc(),
)

ranked = bronze.withColumn(
    "_current_rn",
    F.row_number().over(latest_window),
)

current_listings = (
    ranked
    .filter(F.col("_current_rn") == 1)
    .filter(F.col("_pg_change_type").isin("insert", "update_postimage"))
    .filter(F.col("listing_id").isNotNull())
    .filter(F.col("latitude").between(-90, 90))
    .filter(F.col("longitude").between(-180, 180))
    .select(
        "listing_id",
        "donor_id",
        "title",
        "description",
        "category",
        "condition",
        "latitude",
        "longitude",
        "available_from",
        "available_until",
        "status",
        "updated_at",
        F.col("_pg_lsn").alias("source_pg_lsn"),
        F.col("_sort_by").alias("source_sort_by"),
        F.col("_pg_change_type").alias("source_change_type"),
        F.current_timestamp().alias("silver_processed_at"),
    )
)

display(current_listings)


## Validation gate

Do not write the target until these checks pass.


In [ ]:
row_count = current_listings.count()
duplicate_keys = (
    current_listings.groupBy("listing_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_keys = current_listings.filter(F.col("listing_id").isNull()).count()
invalid_coords = current_listings.filter(
    ~F.col("latitude").between(-90, 90)
    | ~F.col("longitude").between(-180, 180)
).count()

print("current rows:", row_count)
print("duplicate listing_id:", duplicate_keys)
print("null listing_id:", null_keys)
print("invalid coordinates:", invalid_coords)

assert duplicate_keys == 0
assert null_keys == 0
assert invalid_coords == 0


In [ ]:
display(
    current_listings.filter(
        F.col("listing_id").isin(
            "demo-listing-001",
            "demo-listing-002",
        )
    )
)


## Write only after validation passes

The overwrite is intentionally idempotent for the capstone current-state table.


In [ ]:
(
    current_listings.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

print("Wrote:", TARGET)
print("Silver listing rows:", spark.table(TARGET).count())
